# fiftyone_review_processed.ipynb — browse CONVERTED (intermediate-schema) data

**When to use this:** *after* Stage 5.2 conversion. Loads a converted source's intermediate-schema output (DEC-046) directly — flat `images/`+`labels/`, canonical class ids, plain YOLO `.txt` labels, no train/val/test split yet. This is the stage Stage 5.3 (Box Audit) and Stage 5.5 (Model-Assisted Curation) actually work on, so this notebook is genuinely useful, not just a sanity check.

**What it can browse (set via `source_key`, cell below):**
- A Stage 5.2 processed source, e.g. `"exdark"`, `"roboflow_pothole_vhmow"` — the original single-source use case.
- `"merged"` — the post-cap, post-merge pool (`dataset/merged/`, Stage 5.6).
- `"final/train"` / `"final/val"` / `"final/test"` — the split output (Stage 5.8).

**Flagged-only mode (optional, `flagged_report_path`):** point it at a `box_audit.py`-style flagged-boxes report (e.g. `dataset/reports/elevator_status_s4lrk_flagged.json`) to load *only* the images with at least one flagged box, with the specific flagged detection(s) marked (`detection.flagged == True`) so they're distinguishable from an image's other, unflagged boxes — this is what Stage 5.3's "isolate real detection boxes from shape-defective ones" review actually needs. Only works against a plain processed-source `source_key` (flagged reports reference that source's own filenames, not `merged`/`final`'s source-prefixed ones).

**Why not FiftyOne's built-in YOLO importer:** `fo.types.YOLOv5Dataset` assumes a `dataset.yaml` + per-split (`train`/`val`/`test`) folder structure. That fits `final/<split>`, but not `dataset/processed/<source>/` or `dataset/merged/` (both deliberately flat, no split yet — DEC-036). Building the FiftyOne dataset directly from `images/`+`labels/` handles all three the same way, one code path instead of two.

**Not for:** raw acquisition-stage exports (`dataset/raw/<source>/`) — use `fiftyone_explore.ipynb` (COCO-style) or `fiftyone_preview.ipynb` (pre-pull) for those instead.

In [ ]:
# Imports
import json
import shutil
import sys
from pathlib import Path


def _find_repo_root(start: Path) -> Path:
    """Walk up from `start` to find the repo root (has config/ + AGENTS.md).

    Needed because notebooks live in notebooks/, not the repo root, and
    Jupyter's working directory depends on how it was launched -- this
    makes the scripts.* import below robust regardless of that.
    """
    for parent in [start, *start.parents]:
        if (parent / "config").is_dir() and (parent / "AGENTS.md").is_file():
            return parent
    raise RuntimeError("Could not locate repo root from notebook cwd.")


REPO_ROOT = _find_repo_root(Path.cwd())
sys.path.insert(0, str(REPO_ROOT))

import fiftyone as fo

from scripts.utils.config_loader import get_canonical_names
from scripts.utils.file_utils import ensure_dir, final_dir, merged_dir, processed_dir

CANONICAL_NAMES = get_canonical_names()

In [ ]:
# Change this and re-run the cells below to browse a different pool.
# Matches dataset/processed/<source_key>/ by default -- e.g. "exdark",
# "dataset_ninja_pothole_detection", "open_images", "roboflow_pothole_vhmow" --
# or the literal strings "merged" (dataset/merged/) or "final/train" /
# "final/val" / "final/test" (dataset/final/<split>/).
source_key = "roboflow_elevator_status_s4lrk"

# Optional: path to a box_audit.py-style flagged-boxes report (list of
# {label_path, class, cx, cy, w, h, reasons}). Set to None for normal
# browsing of every image in source_key. Only valid when source_key is a
# plain processed source (not "merged"/"final/*").
flagged_report_path = REPO_ROOT / "dataset/reports/elevator_status_s4lrk_flagged.json"

In [ ]:
# Build the FiftyOne dataset directly from images/+labels/ -- no network
# calls, safe to re-run anytime (re-running replaces the same dataset from
# scratch, discarding any in-App edits made since the last run).
#
# Persistence note: persistent=False means this dataset's edits do NOT
# survive a kernel restart (or fo.delete_dataset), but they DO auto-save
# to FiftyOne's backing DB *during* this session -- editing/adding/deleting
# a box via the App's "Annotate" tab is real and immediately reflected in
# `dataset` here, it just isn't durable across sessions on its own. A
# separate write-back step (see notebook README / ask before building) is
# what would turn a review session into changes on disk in dataset/processed/.

# Resolve images_dir/labels_dir for the chosen pool. "merged" and "final/<split>"
# use file_utils' own path helpers (a different directory layout, source-prefixed
# filenames); everything else is treated as a Stage 5.2 processed source key.
if source_key == "merged":
    images_dir = merged_dir() / "images"
    labels_dir = merged_dir() / "labels"
elif source_key.startswith("final/"):
    split = source_key.split("/", 1)[1]
    images_dir = final_dir(split) / "images"
    labels_dir = final_dir(split) / "labels"
else:
    images_dir = processed_dir(source_key) / "images"
    labels_dir = processed_dir(source_key) / "labels"

# Optional flagged-only mode: restrict to images with >=1 flagged box, and mark
# which specific detection(s) triggered the flag -- and *why* (box_audit.py's
# reasons, e.g. "large_area_outlier (>0.31)" -- a near-full-image box, the
# classic classification-dataset-forced-into-detection symptom) -- so they
# stand out from an image's other, unflagged boxes and you're not guessing
# why something was flagged.
flagged_by_label_path: dict[str, list[dict]] = {}
if flagged_report_path is not None:
    if source_key == "merged" or source_key.startswith("final/"):
        raise ValueError(
            "flagged_report_path is only supported against a plain processed-source "
            "source_key -- merged/final use prefixed filenames a flagged report doesn't reference."
        )
    flagged_entries = json.loads(Path(flagged_report_path).read_text(encoding="utf-8"))
    for entry in flagged_entries:
        flagged_by_label_path.setdefault(entry["label_path"], []).append(entry)

dataset_name = f"review_{source_key.replace('/', '_')}"
if dataset_name in fo.list_datasets():
    fo.delete_dataset(dataset_name)
dataset = fo.Dataset(dataset_name, persistent=False)

samples = []
for image_path in sorted(images_dir.iterdir()):
    label_path = labels_dir / f"{image_path.stem}.txt"
    label_filename = label_path.name

    if flagged_by_label_path and label_filename not in flagged_by_label_path:
        continue  # flagged mode: skip images with nothing flagged

    # Match by rounded (cx, cy, w, h), not raw float equality -- both sides come
    # from the same 6-decimal string formatting this project's converters use,
    # but comparing through a round-trip is more robust than trusting exact
    # float equality to hold.
    flagged_lookup = {
        (round(e["cx"], 6), round(e["cy"], 6), round(e["w"], 6), round(e["h"], 6)): e["reasons"]
        for e in flagged_by_label_path.get(label_filename, [])
    }

    sample = fo.Sample(filepath=str(image_path))
    # Stashed so a future write-back step knows exactly which label file a
    # sample's (possibly since-edited) detections came from, without having
    # to re-derive it from the image filename.
    sample["source_label_filename"] = label_filename
    detections = []
    if label_path.is_file():
        for line in label_path.read_text(encoding="utf-8").splitlines():
            parts = line.strip().split()
            if not parts:
                continue
            class_id = int(parts[0])
            cx, cy, w, h = (float(v) for v in parts[1:5])
            # Our labels are YOLO center-based (cx, cy, w, h); FiftyOne's
            # Detection.bounding_box is top-left-based (x, y, w, h) --
            # both normalized [0, 1], so just shift the origin.
            x, y = cx - w / 2, cy - h / 2
            reasons = flagged_lookup.get((round(cx, 6), round(cy, 6), round(w, 6), round(h, 6)))
            detections.append(
                fo.Detection(
                    label=CANONICAL_NAMES[class_id],
                    bounding_box=[x, y, w, h],
                    flagged=reasons is not None,
                    flag_reasons=", ".join(reasons) if reasons else "",
                )
            )

    sample["ground_truth"] = fo.Detections(detections=detections)
    samples.append(sample)

dataset.add_samples(samples)
print(f"{len(dataset)} images loaded from {images_dir}")
if flagged_by_label_path:
    total_flagged_boxes = sum(len(v) for v in flagged_by_label_path.values())
    print(
        f"Flagged-only mode: {len(flagged_by_label_path)} images, {total_flagged_boxes} flagged boxes "
        f"-- marked detections have flagged == True, with the reason(s) on flag_reasons "
        f"(visible in the App's sample modal, under the detection's attributes)"
    )

In [ ]:
# Launch the App
session = fo.launch_app(dataset, auto=False)

In [ ]:
session

# Write back your review edits

**Run this only after you're done editing in the App above, in the same kernel session** (it reads the live `dataset` object — restarting the kernel loses everything, since this dataset is `persistent=False`).

This does **not** touch your original files in `dataset/processed/<source>/labels/`. It writes to a parallel `labels_reviewed/` folder instead, plus prints a per-file summary of what changed, so you can spot-check before deciding to promote anything. Promoting (copying `labels_reviewed/*.txt` over the real `labels/`) is a separate, deliberate step — not automatic — because this write-back path hasn't been used for a real correction pass yet.

Only valid for a plain processed-source `source_key` (same restriction as `flagged_report_path` above) — `merged`/`final` are derived outputs regenerated by other scripts, not something to hand-edit here.

In [ ]:
if source_key == "merged" or source_key.startswith("final/"):
    raise ValueError(
        "Write-back is only supported against a plain processed-source source_key -- "
        "merged/final are derived outputs regenerated by other scripts, not hand-edited here."
    )

orig_labels_dir = processed_dir(source_key) / "labels"
out_labels_dir = processed_dir(source_key) / "labels_reviewed"
if out_labels_dir.is_dir():
    shutil.rmtree(out_labels_dir)  # staging area only -- safe to clear and rewrite each run
ensure_dir(out_labels_dir)

added = removed = modified = unchanged = 0
for sample in dataset:
    label_filename = sample["source_label_filename"]
    orig_path = orig_labels_dir / label_filename
    orig_lines = [
        ln.strip() for ln in (orig_path.read_text(encoding="utf-8").splitlines() if orig_path.is_file() else [])
        if ln.strip()
    ]

    new_lines = []
    for det in sample.ground_truth.detections:
        if det.label not in CANONICAL_NAMES:
            raise ValueError(
                f"{label_filename}: detection has label {det.label!r}, not one of the 16 canonical "
                f"classes -- check for a typo introduced via the App's class dropdown before re-running."
            )
        class_id = CANONICAL_NAMES.index(det.label)
        # Inverse of the load cell's transform: FiftyOne's top-left [x, y, w, h] -> YOLO center-based.
        x, y, w, h = det.bounding_box
        cx, cy = x + w / 2, y + h / 2
        new_lines.append(f"{class_id} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")

    (out_labels_dir / label_filename).write_text(
        "\n".join(new_lines) + ("\n" if new_lines else ""), encoding="utf-8"
    )

    if len(new_lines) > len(orig_lines):
        added += 1
    elif len(new_lines) < len(orig_lines):
        removed += 1
    elif set(new_lines) != set(orig_lines):
        modified += 1
    else:
        unchanged += 1

print(f"Wrote {len(dataset)} label files to {out_labels_dir}")
print(
    f"vs. originals in {orig_labels_dir}: "
    f"{added} files gained box(es), {removed} files lost box(es), "
    f"{modified} files changed geometry/class only (same count), {unchanged} untouched"
)
print(
    "\nNothing in dataset/processed/<source>/labels/ has been touched yet. Spot-check "
    "labels_reviewed/ (e.g. re-point source_key's flagged_report_path-free run at it, or diff "
    "a few files by hand), then explicitly copy the ones you're confident in over the real "
    "labels/ folder when ready to promote -- that promotion step is intentionally manual."
)